# 05 — Recovering the hair-cell layer (route C)SHD ships only the last stage of the LAUSCHER chain:```audio -> basilar membrane -> hair cell -> bushy cell -> SHD                             (Meddis)     (LIF, 40 afferents)                             continuous   spikes                             probability```Two facts from the LAUSCHER source make the stage above recoverable:1. The hair-cell stage emits a **firing probability**, not spikes.2. Hair-cell and bushy-cell channels are **one-to-one** — the 40 afferents are stochastic draws *within* a channel, not a fan-in across channels.So this is a per-channel inverse problem with a monotone, saturating forward map. Measure the map, invert it.Read [docs/04-hair-cell-recovery.md](../docs/04-hair-cell-recovery.md) before drawing conclusions from anything here — the limits matter as much as the method.

In [ ]:
%load_ext autoreload%autoreload 2import numpy as npimport matplotlib.pyplot as pltfrom compbio2026 import data, lauscher, plottingplotting.apply_style()rng = np.random.default_rng(2026)

## Step 1 — the forward mapDrive the re-implemented bushy cell with constant probabilities and record the firing rate. This takes a minute or two; cache the result.

In [ ]:
curve = lauscher.transfer_curve(duration=0.2, n_repeats=3)probs, rates = curvefig, ax = plt.subplots(figsize=(6, 3.8))plotting.transfer_curve(probs, rates, ax=ax)print(f"max rate reached: {rates.max():.0f} spikes/s   (refractory ceiling = {1/1e-3:.0f})")

The curve saturates at the refractory ceiling, `1 / tau_refrac = 1000 spikes/s`. Above the knee very different probabilities map to nearly the same rate, so the inversion there is **ill-posed**. Those samples are censored, not confidently high.

## Step 2 — validate on synthetic data, where you know the answerDo this before touching SHD. Pick a smooth `p(t)`, simulate a bushy cell from it, then recover `p` from the spikes and compare.

In [ ]:
fs = 20_000.0duration = 0.5n = int(fs * duration)tt = np.arange(n) / fsp_true = 0.02 * (1 + np.sin(2 * np.pi * 6 * tt)) * np.exp(-((tt - 0.25) ** 2) / 0.02)bc = lauscher.BushyCell(**lauscher.BC_DEFAULTS)spikes = bc.simulate(p_true, fs, seed=0)print(f"{spikes.sum():.0f} spikes in {duration} s")

In [ ]:
bin_ms = 1.0step = int(fs * bin_ms / 1e3)binned = spikes[: (n // step) * step].reshape(1, 1, -1, step).sum(axis=-1)   # (1, 1, bins)p_hat, valid = lauscher.estimate_hair_cell_rate(binned, bin_ms=bin_ms, curve=curve, smooth_ms=8.0)t_bin = (np.arange(p_hat.shape[-1]) + 0.5) * bin_ms / 1e3p_true_binned = p_true[: (n // step) * step].reshape(-1, step).mean(axis=1)fig, ax = plt.subplots(figsize=(9, 3.6))ax.plot(t_bin, p_true_binned, label="true p(t)", color=plotting.INK, lw=1.4)ax.plot(t_bin, p_hat[0, 0], label="recovered", color=plotting.ACCENT, lw=1.4)ax.fill_between(t_bin, 0, p_hat[0, 0].max(), where=~valid[0, 0], color=plotting.INK_MUTED,                alpha=0.12, linewidth=0, label="saturated (censored)")ax.set_xlabel("Time (s)")ax.set_ylabel("Firing probability per sample")ax.legend()r = np.corrcoef(p_true_binned, p_hat[0, 0])[0, 1]print(f"correlation with truth: r = {r:.3f}")print(f"fraction censored     : {(~valid).mean():.3f}")

**Report both numbers.** The correlation says how well the method works; the censored fraction says how much of the signal it could not see. A high correlation on 5 % of the samples is not the same claim as a high correlation on 95 %.Now vary the smoothing kernel and watch the trade-off — this is the parameter that sets your time resolution.

In [ ]:
for s in [2.0, 5.0, 10.0, 25.0]:    ph, va = lauscher.estimate_hair_cell_rate(binned, bin_ms=bin_ms, curve=curve, smooth_ms=s)    print(f"smooth {s:5.1f} ms   r = {np.corrcoef(p_true_binned, ph[0,0])[0,1]:.3f}")

## Step 3 — apply it to SHDSingle trials are sparse: 40 Bernoulli draws per timestep average out a lot, but one utterance does not carry enough spikes to invert finely. **Average over trials of the same digit first** — then the estimate is a statement about the class, not the utterance.

In [ ]:
shd = data.load("train")word = "seven"idx = np.flatnonzero(shd.labels == data.DIGIT_KEYS.index(word))[:60]X, y, t = data.build_design_matrix(shd, bin_ms=2.0, t_max_ms=800.0, trials=idx)X_mean = X.mean(axis=0, keepdims=True)p_hat, valid = lauscher.estimate_hair_cell_rate(X_mean, bin_ms=2.0, curve=curve, smooth_ms=6.0)print("censored fraction:", round(float((~valid).mean()), 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)axes[0].imshow(X_mean[0], aspect="auto", origin="lower", cmap="Greys", extent=[0, 800, 0, 700])axes[0].set_title(f'Bushy cell (SHD), mean of {len(idx)} trials, "{word}"')axes[1].imshow(p_hat[0], aspect="auto", origin="lower", cmap="Greys", extent=[0, 800, 0, 700])axes[1].set_title("Recovered hair-cell firing probability")for ax in axes:    ax.set_xlabel("Time (ms)")axes[0].set_ylabel("Channel (tonotopic)")fig.tight_layout()

## Step 4 — why this is worth doingWith the hair-cell layer recovered you can run **the whole Stage 1–3 analysis on both layers and compare**:- Does the bushy-cell layer separate the digits better than the hair-cell layer it came from?- Or does coincidence detection discard information a decoder could have used?That is a genuinely open question, it is answerable with the tools already in this repository, and it is a complete project on its own.

In [ ]:
# Sketch: compare decodability across layers.# Recover the hair-cell layer for a balanced subset, then decode from each.## from compbio2026 import decoding# X_bc, y_sub, _ = data.build_design_matrix(shd, bin_ms=2.0, trials=subset)# p_hc, _        = lauscher.estimate_hair_cell_rate(X_bc, bin_ms=2.0, curve=curve, smooth_ms=6.0)# print("bushy cell:", decoding.decode(data.flatten(X_bc), y_sub)["mean"])# print("hair cell :", decoding.decode(data.flatten(p_hc), y_sub)["mean"])## Match the readout and the dimensionality across layers, or the comparison# is about feature count rather than about the representation.

---## Exercises1. **Ground truth.** Install LAUSCHER and run `lauscher.run_lauscher()` on a `.wav` from the Heidelberg Digits audio. Compare the true firing probability against your estimate from the bushy-cell spikes of the same run. That closes the loop with no unknowns.2. **The censored region.** What fraction of SHD samples fall in saturation? Are they concentrated in particular channels or particular times? Does that change what you can claim?3. **Convergence.** Re-measure the transfer curve with `n_convergence` in {10, 40, 100}. How does the curve move, and what does that say about how much the recovery depends on knowing the right value?4. **Layer comparison.** Complete the sketch above and report both accuracies with matched readouts and matched dimensionality.